In [1]:
import pandas as pd
import numpy as np
import itertools
import random
from collections import Counter
random.seed(1)


Single Finger Flexion or Extension to a target force. Four different target conditions: +3, +1.5, -1.5, -3 N. 
Each condition has the following structure: 

washout block (all conditinos with gain = 1)

adaptation block (one condition gain = 2)

test block (all conditions tested without visual feedback)

In [2]:
iti = 1000 # msecs for inter-trial interval
planTime = 2000 # msecs for precue time
feedbackTime = 1000 # msecs for feedback time

total_sub_num = 20
num_trials_washout_per_condition = 10
num_trials_adaptation = 40
num_trials_test_per_condition = 15
num_trials_per_block = 32 # or 33 for the last block
num_trials_per_transition = 40

finger = 2  # index (1-based: thumb=1 ... pinky=5)
conditions = [-4, -2, 2, 4]


In [3]:
# target file columns match MyTrial::read:
# subNum, targetForce1-5, isTargetVisible, gain, planTime, feedbackTime, iti
# condition = targetForce2 (index finger)

columns = ['subNum', 'targetForce1', 'targetForce2', 'targetForce3', 'targetForce4', 'targetForce5',
           'isTargetVisible', 'gain', 'planTime', 'feedbackTime', 'iti']

for sub_num in range(1, total_sub_num + 1):
    seed = sub_num
    np.random.seed(seed)
    subj_conditions_order = conditions.copy()
    np.random.shuffle(subj_conditions_order)
    print(subj_conditions_order)
    bn = 1

    for adaptation_condition in subj_conditions_order:
        # washout block
        rows = []
        for condition in conditions:
            target_forces = [0.0] * 5
            target_forces[finger - 1] = condition
            gain = 1.0
            rows.append([
                sub_num,
                *target_forces,
                1,
                gain,
                planTime,
                feedbackTime,
                iti,
            ])
        block_data = pd.DataFrame(rows, columns=columns).astype({
            'subNum': int,
            'gain': float,
            'planTime': int,
            'feedbackTime': int,
            'iti': int,
            'isTargetVisible': int,
        })
        block_data = pd.concat([block_data] * num_trials_washout_per_condition, ignore_index=True)
        # permute the rows
        block_data = block_data.sample(frac=1).reset_index(drop=True)
        block_data.to_csv(f'sn{sub_num}_{bn}.tgt', index=False, sep='\t')
        bn += 1


        # adaptation block
        target_forces = [0.0] * 5
        target_forces[finger - 1] = adaptation_condition
        gain = 2.0
        block_data = pd.DataFrame(
            [[
                sub_num,
                *target_forces,
                1,
                gain,
                planTime,
                feedbackTime,
                iti,
            ]],
            columns=columns,
        )
        block_data = pd.concat([block_data] * num_trials_adaptation, ignore_index=True)
        block_data.to_csv(f'sn{sub_num}_{bn}.tgt', index=False, sep='\t')
        bn += 1


        # test block
        rows = []
        gain = 1.0
        for condition in conditions:
            target_forces = [0.0] * 5
            target_forces[finger - 1] = condition
            rows.append([
                sub_num,
                *target_forces,
                0,
                gain,
                planTime,
                feedbackTime,
                iti,
            ])
        block_data = pd.DataFrame(rows, columns=columns).astype({
            'subNum': int,
            'gain': float,
            'planTime': int,
            'feedbackTime': int,
            'iti': int,
            'isTargetVisible': int,
        })
        block_data = pd.concat([block_data] * num_trials_test_per_condition, ignore_index=True)
        block_data = block_data.sample(frac=1).reset_index(drop=True)
        block_data.to_csv(f'sn{sub_num}_{bn}.tgt', index=False, sep='\t')
        bn += 1



[4, 2, -4, -2]
[2, 4, -2, -4]
[4, -2, -4, 2]
[-4, -2, 4, 2]
[-4, -2, 2, 4]
[-4, 4, -2, 2]
[2, -2, -4, 4]
[2, -2, -4, 4]
[-2, 4, -4, 2]
[2, -4, 4, -2]
[2, 4, -4, -2]
[-4, -2, 2, 4]
[-2, 4, -4, 2]
[-2, 2, -4, 4]
[2, 4, -2, -4]
[-4, 4, 2, -2]
[-4, 2, -2, 4]
[4, -2, -4, 2]
[-4, 4, 2, -2]
[-4, -2, 2, 4]
